<a href="https://colab.research.google.com/github/gnassiwa37-sys/PRACTICE-CODES/blob/main/Coursework2_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coursework 2: Predicting Repeat Purchases at RunAway Sneaker Store
**Python for Data Science** | Nassiwa Gloria

Place `customer_data.csv` in the same folder as this notebook, then choose **Run All**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)

RAW_FILE = "customer_data.csv"
CLEAN_FILE = "customer_data_clean.csv"
SEED = 42
np.random.seed(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__, "| seed", SEED)

## TASK 2(a): Loading the raw data

In [ ]:
df_raw = pd.read_csv(RAW_FILE)
print(df_raw.shape)
print(df_raw.head())
print(df_raw.columns.tolist())
print(df_raw.dtypes)

(300, 13)
  customer_id first_purchase_date   age  gender         location   brand  \
0        C001          02/06/2026  39.0  Female  Kampala Central  Adidas   
1        C002          02/06/2026  23.0  Female           Rubaga  Adidas   
2        C003          02/06/2026  19.0  Female  Kampala Central    Nike   
3        C004          04/06/2026  17.0  Female         Makindye  Adidas   
4        C005          04/06/2026  22.0    Male  Kampala Central  Jordan   

   price_ugx payment_method found_us_via bought_online discount_used  \
0   170000.0   Mobile Money       TikTok           Yes            No   
1   175000.0   Mobile Money      Walk-in            No            No   
2   195000.0   Mobile Money       TikTok           Yes           Yes   
3   170000.0   Mobile Money     Referral            No           Yes   
4   180000.0   Mobile Money     WhatsApp           Yes            No   

   visits_before_buying bought_again  
0                   5.0          Yes  
1                   2.

## TASK 2(b): Profiling

In [ ]:
print("Missing values:\n", df_raw.isna().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
print("Unique counts:\n", df_raw.nunique())
print(df_raw.describe())
print(df_raw["bought_again"].value_counts())

Missing values:
 customer_id             200
first_purchase_date     200
age                     200
gender                  200
location                200
brand                   200
price_ugx               200
payment_method          200
found_us_via            200
bought_online           200
discount_used           200
visits_before_buying    200
bought_again            200
dtype: int64
Duplicate rows: 199
Unique counts:
 customer_id             100
first_purchase_date      58
age                      21
gender                    2
location                  6
brand                     6
price_ugx                11
payment_method            3
found_us_via              6
bought_online             2
discount_used             2
visits_before_buying      6
bought_again              2
dtype: int64
              age      price_ugx  visits_before_buying
count  100.000000     100.000000            100.000000
mean    24.820000  177600.000000              2.080000
std      5.701993   13567.63

## TASK 2(c): Cleaning (one reusable function)

In [ ]:
REQUIRED = ["customer_id", "first_purchase_date", "age", "gender", "location", "brand",
            "price_ugx", "payment_method", "found_us_via", "bought_online",
            "discount_used", "visits_before_buying", "bought_again"]

def clean_data(df):
    missing_cols = [c for c in REQUIRED if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")
    df = df.copy()
    log = {}

    n = len(df); df = df.drop_duplicates().drop_duplicates(subset="customer_id")
    log["duplicates removed"] = n - len(df)

    df["first_purchase_date"] = pd.to_datetime(df["first_purchase_date"], dayfirst=True, errors="coerce")
    for col in ["age", "price_ugx", "visits_before_buying"]:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce")

    yn = {"yes": "Yes", "y": "Yes", "1": "Yes", "true": "Yes", "no": "No", "n": "No", "0": "No", "false": "No"}
    for col in ["bought_online", "discount_used", "bought_again"]:
        df[col] = df[col].astype(str).str.strip().str.lower().map(yn)
    pay = {"mobile money": "Mobile Money", "momo": "Mobile Money", "cash": "Cash", "card": "Card"}
    df["payment_method"] = df["payment_method"].astype(str).str.strip().str.lower().map(pay)

    before = df[["age", "price_ugx", "visits_before_buying"]].isna().sum().sum()
    df.loc[~df["age"].between(10, 100), "age"] = np.nan
    df.loc[df["price_ugx"] <= 0, "price_ugx"] = np.nan
    df.loc[df["visits_before_buying"] < 0, "visits_before_buying"] = np.nan
    log["impossible values set to missing"] = int(df[["age", "price_ugx", "visits_before_buying"]].isna().sum().sum() - before)

    n = len(df); df = df.dropna(subset=["bought_again"])
    log["rows dropped (missing target)"] = n - len(df)

    log["missing predictors filled"] = int(df.drop(columns="bought_again").isna().sum().sum())
    for col in df.columns.drop("bought_again"):
        if df[col].dtype.kind in "if":
            df[col] = df[col].fillna(df[col].median())
        elif df[col].dtype.kind != "M":
            df[col] = df[col].fillna(df[col].mode()[0])

    q1, q3 = df["price_ugx"].quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    log["extreme prices capped"] = int(((df["price_ugx"] < lo) | (df["price_ugx"] > hi)).sum())
    df["price_ugx"] = df["price_ugx"].clip(lo, hi)
    return df.reset_index(drop=True), log

df_clean, log = clean_data(df_raw)
print("Cleaning log:", log)

## TASK 2(d): Transformations

In [ ]:
def build_features(df):
    df = df.copy()
    for col in ["bought_online", "discount_used", "bought_again"]:
        df[col] = (df[col] == "Yes").astype(int)
    df["social_channel"] = df["found_us_via"].isin(["Instagram", "TikTok", "WhatsApp"]).astype(int)
    df["digital_score"] = df["bought_online"] + df["social_channel"] + df["discount_used"]
    df["purchase_day_of_month"] = df["first_purchase_date"].dt.day
    df["weekend_purchase"] = (df["first_purchase_date"].dt.dayofweek >= 5).astype(int)
    df = df.drop(columns=["customer_id", "first_purchase_date"])
    cats = ["gender", "location", "brand", "payment_method", "found_us_via"]
    return pd.get_dummies(df, columns=cats, drop_first=True, dtype=int)

df_enc = build_features(df_clean)
print("Cleaned:", df_clean.shape, "| Encoded:", df_enc.shape)

Cleaned: (100, 13) | Encoded: (100, 28)


## TASK 2(e)/(f): Before and after, save clean file

In [ ]:
print("Before:", df_raw.shape, df_raw.isna().sum().sum(), df_raw.duplicated().sum())
print("After :", df_clean.shape, df_clean.isna().sum().sum(), df_clean.duplicated().sum())
print(df_clean.info()); print(df_clean.describe()); print(df_clean.sample(5, random_state=SEED))
df_clean.to_csv(CLEAN_FILE, index=False)          # raw file is never edited

# Figure 1: missing values + target distribution
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df_raw.isna().sum().plot.barh(ax=ax[0], color="#8D99AE", title="Missing values per column")
df_clean["bought_again"].value_counts().plot.bar(ax=ax[1], color=["#8D99AE", "#FF5A1F"], title="Target: bought_again")
plt.tight_layout(); plt.savefig("fig1_missing_target.png", dpi=150); plt.close()

# Figure 2: correlation of numeric features
num = df_enc[["age", "price_ugx", "visits_before_buying", "digital_score", "bought_again"]]
plt.figure(figsize=(6, 4.5)); plt.imshow(num.corr(), cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(num.columns)), num.columns, rotation=45, ha="right"); plt.yticks(range(len(num.columns)), num.columns)
plt.title("Correlation between numeric features"); plt.tight_layout(); plt.savefig("fig2_correlation.png", dpi=150); plt.close()

## TASK 3: Algorithm and model development

In [ ]:
class Standardiser:
    def fit(self, X):
        self.mean, self.std = X.mean(axis=0), X.std(axis=0)
        self.std[self.std == 0] = 1.0
        return self
    def transform(self, X):
        return (X - self.mean) / self.std

class LogisticRegressionGD:
    def __init__(self, lr=0.1, n_iter=2000, l2=0.01):
        if lr <= 0 or n_iter <= 0 or l2 < 0:
            raise ValueError("Bad parameters")
        self.lr, self.n_iter, self.l2 = lr, n_iter, l2
        self.w, self.b, self.loss_history = None, 0.0, []
    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
    def fit(self, X, y):
        if X.shape[0] != y.shape[0] or np.isnan(X).any():
            raise ValueError("Shape mismatch or NaNs")
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        for _ in range(self.n_iter):
            p = self._sigmoid(X @ self.w + self.b)
            self.w -= self.lr * ((X.T @ (p - y)) / n + self.l2 * self.w)
            self.b -= self.lr * np.mean(p - y)
            eps = 1e-12
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)) + 0.5 * self.l2 * np.sum(self.w ** 2)
            self.loss_history.append(loss)
        return self
    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

def evaluate(y_true, y_pred, y_prob):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.5}

In [ ]:
# --- 3(d): split, scale, train, compare ---
X = df_enc.drop(columns="bought_again").astype(float)
y = df_enc["bought_again"].values
feature_names = X.columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X.values, y, test_size=0.2, stratify=y, random_state=SEED)
sc = Standardiser().fit(X_tr)                      # fitted on TRAIN only (no leakage)
X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)
print("Train", X_tr.shape, "Test", X_te.shape)

own = LogisticRegressionGD(lr=0.1, n_iter=2000, l2=0.01).fit(X_tr_s, y_tr)
skl = LogisticRegression(C=1 / (0.01 * len(y_tr)), max_iter=2000).fit(X_tr_s, y_tr)
rf = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_tr, y_tr)
majority = int(y_tr.mean() >= 0.5)

results = pd.DataFrame({
    "Baseline (majority)": evaluate(y_te, np.full_like(y_te, majority), np.full(len(y_te), 0.5)),
    "Own logistic regression": evaluate(y_te, own.predict(X_te_s), own.predict_proba(X_te_s)),
    "sklearn logistic regression": evaluate(y_te, skl.predict(X_te_s), skl.predict_proba(X_te_s)[:, 1]),
    "Random Forest": evaluate(y_te, rf.predict(X_te), rf.predict_proba(X_te)[:, 1]),
}).T.round(2)
print(results)

In [ ]:
# --- 5-fold stratified cross-validation of own model ---
accs, f1s = [], []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X.values, y):
    s = Standardiser().fit(X.values[tr])
    m = LogisticRegressionGD().fit(s.transform(X.values[tr]), y[tr])
    pred = m.predict(s.transform(X.values[va]))
    accs.append(accuracy_score(y[va], pred)); f1s.append(f1_score(y[va], pred, zero_division=0))
print(f"CV accuracy {np.mean(accs):.3f} +/- {np.std(accs):.3f} | CV F1 {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")

In [ ]:
# --- strongest coefficients ---
coef = pd.Series(own.w, index=feature_names).sort_values()
top = pd.concat([coef.head(5), coef.tail(5)])
print(top)

## Figures 3-6

In [ ]:
plt.figure(figsize=(6, 4)); plt.plot(own.loss_history); plt.xlabel("Iteration"); plt.ylabel("Loss")
plt.title("Training loss (gradient descent)"); plt.tight_layout(); plt.savefig("fig3_loss.png", dpi=150); plt.close()

cm = confusion_matrix(y_te, own.predict(X_te_s))
plt.figure(figsize=(4.5, 4)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
for (i, j), v in np.ndenumerate(cm): plt.text(j, i, v, ha="center", va="center")
plt.xticks([0, 1], ["No", "Yes"]); plt.yticks([0, 1], ["No", "Yes"]); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix (own model)"); plt.tight_layout(); plt.savefig("fig4_confusion.png", dpi=150); plt.close()

plt.figure(figsize=(6, 5))
for name, prob in [("Own LR", own.predict_proba(X_te_s)), ("sklearn LR", skl.predict_proba(X_te_s)[:, 1]),
                   ("Random Forest", rf.predict_proba(X_te)[:, 1])]:
    fpr, tpr, _ = roc_curve(y_te, prob); plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], "k--", label="Chance"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves"); plt.legend(); plt.tight_layout(); plt.savefig("fig5_roc.png", dpi=150); plt.close()

plt.figure(figsize=(7, 5)); top.plot.barh(color=["#C0504D" if v < 0 else "#4F81BD" for v in top])
plt.title("Strongest coefficients (own model, standardised features)"); plt.tight_layout()
plt.savefig("fig6_coefficients.png", dpi=150); plt.close()
print("Done. Figures saved as fig1..fig6 .png")